mount google drive



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import geopandas as gpd, pandas as pd
t = pd.read_csv('singapore_tiles.csv')
print("candidate tiles:", len(t))
print("filtered tiles:", len(t[~((t.n_amenities==0)&(t.n_intersections==0)&(t.landuse_entropy==0))]))
print("total amenities across tiles:", int(t.n_amenities.sum()))
print("total intersections across tiles:", int(t.n_intersections.sum()))

In [ ]:

"""
make_figure3.py  -- Figure 3, training dynamics of both models.
Run in Colab after mounting Drive. Produces a single-column (3.4 in) figure
at 600 dpi, suitable for the template.
"""
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DRIVE = "/content/drive/MyDrive/pix2pix_runs"
cond = pd.read_csv(f"{DRIVE}/conditional_losses.csv")
base = pd.read_csv(f"{DRIVE}/baseline_losses.csv")

# SAFETY CHECK: make sure this is the RETRAINED baseline, not the collapsed run.
alive = (base.D > 0.01).sum()
print(f"baseline discriminator active in {alive}/{len(base)} epochs")
assert alive > 100, ("This looks like the COLLAPSED baseline run. Use the "
                     "retrained file (D active in ~192/200 epochs).")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.4, 4.2), sharex=True)

ax1.plot(cond.epoch, cond.G, linewidth=0.9, label="Conditional")
ax1.plot(base.epoch, base.G, linewidth=0.9, linestyle="--", label="Unconditional")
ax1.set_ylabel("Generator loss", fontsize=8)
ax1.legend(fontsize=7, frameon=False)
ax1.tick_params(labelsize=7)

ax2.plot(cond.epoch, cond.D, linewidth=0.9, label="Conditional")
ax2.plot(base.epoch, base.D, linewidth=0.9, linestyle="--", label="Unconditional")
ax2.set_ylabel("Discriminator loss", fontsize=8)
ax2.set_xlabel("Epoch", fontsize=8)
ax2.tick_params(labelsize=7)

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"{DRIVE}/figure3_losses.png", dpi=600, bbox_inches="tight")
plt.savefig("figure3_losses.png", dpi=600, bbox_inches="tight")
print("saved figure3_losses.png to Drive and to the session")

pull prevous files from script 01

In [ ]:
!cp /content/drive/MyDrive/AI-UrbanPlanning/singapore_tiles.csv .
!cp /content/drive/MyDrive/AI-UrbanPlanning/singapore_tiles.gpkg .
!ls -lh singapore_tiles.*

install packages

In [ ]:
!pip install osmnx geopandas rasterio shapely --quiet

script 03

In [ ]:
"""
03_filter_and_check.py
======================
Applies the exclusion decision officially and re-validates the design on the
filtered dataset. Produces the tile list that ALL downstream scripts use.

Exclusion criterion (mentor-approved): tiles with zero amenities AND zero
intersections AND zero land-use entropy — non-habitable land (industrial
estates, military training areas, water catchment), verified by map
inspection on 16 Jul.

Inputs : singapore_tiles.csv          (from script 01)
Outputs: singapore_tiles_filtered.csv (official training tile list, with labels)
         filtered_stats.txt           (numbers for the paper's Data section)

Run AFTER mounting Drive. Copies outputs to Drive automatically.
"""

import os, shutil
import numpy as np
import pandas as pd

df = pd.read_csv("singapore_tiles.csv")
n_full = len(df)

# ---- exclusion ----------------------------------------------------------
floor = (df["n_amenities"] == 0) & (df["n_intersections"] == 0) & \
        (df["landuse_entropy"] == 0)
f = df[~floor].copy().reset_index(drop=True)
print(f"full: {n_full} tiles | excluded (floor): {floor.sum()} | kept: {len(f)}")

# ---- re-standardize composites on the filtered population ---------------
for c in ["walkability", "flood_resilience"]:
    f[c] = (f[c] - f[c].mean()) / f[c].std(ddof=0)

# ---- correlations -------------------------------------------------------
cols = ["walkability", "flood_resilience", "density"]
pearson = f[cols].corr()
spearman = f[cols].corr(method="spearman")
r = pearson.loc["walkability", "flood_resilience"]

# ---- labels (tercile, tie-safe) -----------------------------------------
for c in ["walkability", "flood_resilience"]:
    f[c + "_label"] = pd.qcut(f[c].rank(method="first"), q=3,
                              labels=["low", "med", "high"])
ct = pd.crosstab(f["walkability_label"], f["flood_resilience_label"])

# ---- report -------------------------------------------------------------
lines = [
    f"Full dataset: {n_full} tiles",
    f"Excluded (zero amenities+intersections+entropy): {int(floor.sum())}",
    f"Filtered training dataset: {len(f)} tiles",
    "",
    "Pearson:",  pearson.round(3).to_string(),
    "Spearman:", spearman.round(3).to_string(),
    "",
    f"MAIN CHECK walkability vs flood_resilience: r = {r:.3f}",
    "",
    "Label crosstab:", ct.to_string(),
    f"smallest cell: {ct.values.min()}",
]
report = "\n".join(lines)
print("\n" + report)

f.to_csv("singapore_tiles_filtered.csv", index=False)
with open("filtered_stats.txt", "w") as fh:
    fh.write(report + "\n")
print("\nSaved singapore_tiles_filtered.csv and filtered_stats.txt")

DRIVE = "/content/drive/MyDrive"
if os.path.isdir(DRIVE):
    for p in ["singapore_tiles_filtered.csv", "filtered_stats.txt"]:
        shutil.copy(p, os.path.join(DRIVE, p))
    print("Copied to Drive.")
else:
    print("WARNING: Drive not mounted — copy the outputs manually.")

In [ ]:
import pandas as pd
t = pd.read_csv('singapore_tiles.csv')
f = t[~((t.n_amenities==0)&(t.n_intersections==0)&(t.landuse_entropy==0))]
print("walkability terciles:", f.walkability.quantile([1/3, 2/3]).round(3).tolist())
print("flood terciles:", f.flood_resilience.quantile([1/3, 2/3]).round(3).tolist())

script 04

In [ ]:
"""
04_build_training_pairs.py
==========================
Builds the image dataset the model trains on. For each of the 1,754 filtered
tiles it renders TWO 256x256 images:

  INPUT canvas  = fixed geography the model designs around: roads + water
  TARGET layout = the real layout: green, water, buildings colored by
                  land-use class, roads

and does the SPATIAL train/test split (contiguous 2km superblocks held out,
~15% of tiles) BEFORE any augmentation. Augmentation itself happens
on-the-fly in the training script (05), applied to train tiles only.

Inputs : singapore_tiles_filtered.csv (script 03), singapore_tiles.gpkg (script 01)
Outputs: dataset/{train,test}/{input,target}/tile_<id>.png
         dataset/manifest.csv   (tile_id, split, labels, scores)
         dataset/preview.png    (6 sample pairs for visual verification)
         dataset.zip            (copied to Drive)

Colab: mount Drive first; pull singapore_tiles_filtered.csv, singapore_tiles.gpkg
from Drive into the working dir; then run. Expect 30-60 min (OSM re-download
is the slow part; osmnx caches it).
"""

import os, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from rasterio import features
from rasterio.transform import from_bounds
from PIL import Image

# ---------------------------------------------------------------- settings
CRS = "EPSG:3414"
IMG = 256                    # image size in pixels (500m tile -> ~2m/pixel)
ROAD_BUFFER_M = 4.0          # half-width to make road lines visible
SUPERBLOCK = 4               # split unit = 4x4 tiles (2km x 2km chunks)
TEST_FRACTION = 0.15
SEED = 42

# class palettes: index 0 is background/black
INPUT_CLASSES  = {"water": 1, "road": 2}
INPUT_PALETTE  = {0: (0,0,0), 1: (0,90,255), 2: (140,140,140)}
TARGET_CLASSES = {"green":1, "water":2, "residential":3, "commercial":4,
                  "industrial":5, "other_building":6, "road":7}
TARGET_PALETTE = {0:(0,0,0), 1:(0,170,0), 2:(0,90,255), 3:(255,190,0),
                  4:(230,30,30), 5:(150,60,200), 6:(240,240,240),
                  7:(140,140,140)}

# ---------------------------------------------------------------- load tiles
print("Loading tile list...")
meta = pd.read_csv("singapore_tiles_filtered.csv")
geoms = gpd.read_file("singapore_tiles.gpkg")[["tile_id", "geometry"]]
tiles = geoms.merge(meta, on="tile_id", how="inner")
tiles = gpd.GeoDataFrame(tiles, geometry="geometry", crs=CRS)
print(f"  {len(tiles)} filtered tiles with geometry")

# ---------------------------------------------------------------- OSM layers
print("Downloading OSM layers (cached if run before)...")
def polys(gdf):
    g = gdf[gdf.geometry.geom_type.isin(["Polygon","MultiPolygon"])]
    return g.to_crs(CRS)[["geometry"]] if len(g) else g

buildings = ox.features_from_place("Singapore", tags={"building": True})
buildings = buildings[buildings.geometry.geom_type.isin(["Polygon","MultiPolygon"])]
buildings = buildings.to_crs(CRS)[["geometry"]].reset_index(drop=True)

green = polys(ox.features_from_place("Singapore", tags={
    "leisure": ["park","garden","nature_reserve","pitch","golf_course"],
    "landuse": ["grass","forest","meadow","recreation_ground","cemetery",
                "farmland","orchard"],
    "natural": ["wood","scrub","grassland","wetland"]}))
water = polys(ox.features_from_place("Singapore", tags={
    "natural":["water"], "landuse":["reservoir","basin"],
    "waterway":["riverbank","canal","river"]}))
landuse = ox.features_from_place("Singapore", tags={
    "landuse":["residential","commercial","industrial","retail"]})
landuse = landuse[landuse.geometry.geom_type.isin(["Polygon","MultiPolygon"])]
landuse = landuse.to_crs(CRS)[["geometry","landuse"]].reset_index(drop=True)

G = ox.graph_from_place("Singapore", network_type="drive")
edges = ox.graph_to_gdfs(G, nodes=False).to_crs(CRS)[["geometry"]]
roads = edges.copy()
roads["geometry"] = roads.geometry.buffer(ROAD_BUFFER_M)
print(f"  buildings {len(buildings):,} | green {len(green):,} | "
      f"water {len(water):,} | roads {len(roads):,}")

# ------------------------------------------------- classify buildings by use
print("Classifying buildings by land use (spatial join)...")
landuse["lu"] = landuse["landuse"].replace({"retail": "commercial"})
bj = gpd.sjoin(buildings, landuse[["geometry","lu"]],
               how="left", predicate="intersects")
bj = bj[~bj.index.duplicated(keep="first")]           # one class per building
buildings["cls"] = bj["lu"].fillna("other_building").map(
    {"residential":"residential", "commercial":"commercial",
     "industrial":"industrial", "other_building":"other_building"})
print(buildings["cls"].value_counts().to_string())

# ---------------------------------------------------------------- rendering
def render_tile(bounds, layers):
    """layers = list of (geoms, class_index), drawn in order (later on top).
    Returns a uint8 class-index raster of shape (IMG, IMG)."""
    transform = from_bounds(*bounds, IMG, IMG)
    out = np.zeros((IMG, IMG), dtype=np.uint8)
    for geoms, idx in layers:
        if len(geoms) == 0:
            continue
        features.rasterize(((g, idx) for g in geoms), out=out,
                           transform=transform, default_value=idx)
    return out

def to_png(arr, palette, path):
    rgb = np.zeros((IMG, IMG, 3), dtype=np.uint8)
    for idx, col in palette.items():
        rgb[arr == idx] = col
    Image.fromarray(rgb).save(path)

def subset(layer, tile_geom):
    idx = layer.sindex.query(tile_geom, predicate="intersects")
    return layer.geometry.iloc[idx]

# --------------------------------------------------------- spatial split
# superblock id from tile grid position; whole 2km blocks go to test.
minx, miny = tiles.total_bounds[0], tiles.total_bounds[1]
cx = ((tiles.geometry.centroid.x - minx) // 500).astype(int)
cy = ((tiles.geometry.centroid.y - miny) // 500).astype(int)
tiles["sblock"] = (cx // SUPERBLOCK).astype(str) + "_" + (cy // SUPERBLOCK).astype(str)

rng = np.random.default_rng(SEED)
blocks = tiles["sblock"].unique()
rng.shuffle(blocks)
test_blocks, count, target_n = set(), 0, int(len(tiles) * TEST_FRACTION)
for b in blocks:
    if count >= target_n:
        break
    test_blocks.add(b)
    count += (tiles["sblock"] == b).sum()
tiles["split"] = np.where(tiles["sblock"].isin(test_blocks), "test", "train")
print(f"\nsplit: {(tiles['split']=='train').sum()} train / "
      f"{(tiles['split']=='test').sum()} test "
      f"({(tiles['split']=='test').mean():.1%})")
print("test-set label coverage:")
print(pd.crosstab(tiles.loc[tiles.split=='test','walkability_label'],
                  tiles.loc[tiles.split=='test','flood_resilience_label']))

# ---------------------------------------------------------------- main loop
for sp in ["train", "test"]:
    for kind in ["input", "target"]:
        os.makedirs(f"dataset/{sp}/{kind}", exist_ok=True)

b_by_cls = {c: buildings[buildings["cls"] == c] for c in
            ["residential", "commercial", "industrial", "other_building"]}

print("\nRendering tiles...")
for i, row in tiles.iterrows():
    tg, bounds = row.geometry, row.geometry.bounds
    w  = subset(water, tg);  gr = subset(green, tg);  rd = subset(roads, tg)

    inp = render_tile(bounds, [(w, INPUT_CLASSES["water"]),
                               (rd, INPUT_CLASSES["road"])])
    layers = [(gr, TARGET_CLASSES["green"]), (w, TARGET_CLASSES["water"])]
    for c in ["residential", "commercial", "industrial", "other_building"]:
        layers.append((subset(b_by_cls[c], tg), TARGET_CLASSES[c]))
    layers.append((rd, TARGET_CLASSES["road"]))
    tgt = render_tile(bounds, layers)

    tid = int(row["tile_id"])
    to_png(inp, INPUT_PALETTE,  f"dataset/{row['split']}/input/tile_{tid}.png")
    to_png(tgt, TARGET_PALETTE, f"dataset/{row['split']}/target/tile_{tid}.png")
    if i % 200 == 0:
        print(f"  {i}/{len(tiles)}")

# ---------------------------------------------------------------- manifest
lab = {"low": 0, "med": 1, "high": 2}
man = tiles[["tile_id","split","walkability","flood_resilience","density",
             "walkability_label","flood_resilience_label"]].copy()
man["walk_lab"]  = man["walkability_label"].map(lab)
man["flood_lab"] = man["flood_resilience_label"].map(lab)
man.drop(columns="geometry", errors="ignore").to_csv("dataset/manifest.csv", index=False)

# ---------------------------------------------------------------- preview
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sample = tiles[tiles.split=="train"].sample(6, random_state=1)
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for j, (_, r) in enumerate(sample.iterrows()):
    tid = int(r["tile_id"])
    axes[0,j].imshow(Image.open(f"dataset/train/input/tile_{tid}.png"))
    axes[0,j].set_title(f"input {tid}", fontsize=9)
    axes[1,j].imshow(Image.open(f"dataset/train/target/tile_{tid}.png"))
    axes[1,j].set_title(f"target w={r['walkability_label']} "
                        f"f={r['flood_resilience_label']}", fontsize=9)
for ax in axes.flat: ax.set_axis_off()
plt.tight_layout(); plt.savefig("dataset/preview.png", dpi=110)
print("Saved dataset/, manifest.csv, preview.png")

# ---------------------------------------------------------------- zip+Drive
shutil.make_archive("dataset", "zip", "dataset")
DRIVE = "/content/drive/MyDrive"
if os.path.isdir(DRIVE):
    shutil.copy("dataset.zip", os.path.join(DRIVE, "dataset.zip"))
    print("dataset.zip copied to Drive.")
else:
    print("WARNING: Drive not mounted — copy dataset.zip manually.")

Pull data from script 04 - the training dataset

In [ ]:
!cp /content/drive/MyDrive/dataset.zip . && unzip -q dataset.zip -d dataset

Script 05: training of GAN model, conditional & baseline

In [ ]:
"""
05_train_pix2pix.py  (rare-checkpoint version — USE THIS ONE)
=============================================================
Trains the conditional Pix2Pix model (BASELINE=False) or the unconditional
baseline (BASELINE=True). Run it twice, once per setting.

STORAGE-SAFE CHECKPOINTING:
  - saves every 50 epochs to ONE file that gets OVERWRITTEN each time
    ({run}_latest.pt), so total footprint never exceeds ~2 model files
  - resumes automatically from {run}_latest.pt if the session dies
  - at the end saves {run}_final.pt + loss CSV + final sample grid

Before running: Runtime > Change runtime type > T4 GPU. Then:
    from google.colab import drive; drive.mount('/content/drive')
    !cp /content/drive/MyDrive/dataset.zip . && unzip -q dataset.zip -d dataset
Paste this script and run. ~3.5-4h for 200 epochs on a T4.
"""

import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ---------------------------------------------------------------- settings
BASELINE   = True          # True = unconditional baseline (labels zeroed)
EPOCHS     = 200
BATCH      = 16
LR         = 2e-4
L1_WEIGHT  = 100.0
SAVE_EVERY = 50             # rare checkpoint, single overwritten file
DATA_DIR   = "dataset"
DRIVE_DIR  = "/content/drive/MyDrive/pix2pix_runs"
RUN_NAME   = "baseline" if BASELINE else "conditional"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU! Runtime > Change runtime type > T4 GPU"
os.makedirs(DRIVE_DIR, exist_ok=True)
torch.manual_seed(42); np.random.seed(42)
LATEST = f"{DRIVE_DIR}/{RUN_NAME}_latest.pt"

# ---------------------------------------------------------------- dataset
class TileDataset(Dataset):
    def __init__(self, split):
        man = pd.read_csv(f"{DATA_DIR}/manifest.csv")
        self.rows = man[man["split"] == split].reset_index(drop=True)
        self.split = split
    def __len__(self): return len(self.rows)
    def _load(self, path):
        arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
        return torch.from_numpy(arr / 127.5 - 1.0).permute(2, 0, 1)
    def __getitem__(self, i):
        r = self.rows.iloc[i]; tid = int(r["tile_id"])
        x = self._load(f"{DATA_DIR}/{self.split}/input/tile_{tid}.png")
        y = self._load(f"{DATA_DIR}/{self.split}/target/tile_{tid}.png")
        lab = torch.zeros(6)
        lab[int(r["walk_lab"])] = 1.0
        lab[3 + int(r["flood_lab"])] = 1.0
        if BASELINE: lab = torch.zeros(6)
        if self.split == "train":
            k = np.random.randint(4)
            if k: x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (1, 2))
            if np.random.rand() < 0.5:
                x, y = torch.flip(x, (2,)), torch.flip(y, (2,))
        return x, y, lab, tid

def with_labels(img, lab):
    B, _, H, W = img.shape
    return torch.cat([img, lab.view(B, 6, 1, 1).expand(B, 6, H, W)], dim=1)

# ---------------------------------------------------------------- models
def down(cin, cout, norm=True):
    l = [nn.Conv2d(cin, cout, 4, 2, 1, bias=not norm)]
    if norm: l.append(nn.BatchNorm2d(cout))
    l.append(nn.LeakyReLU(0.2)); return nn.Sequential(*l)

def up(cin, cout, drop=False):
    l = [nn.ConvTranspose2d(cin, cout, 4, 2, 1, bias=False),
         nn.BatchNorm2d(cout), nn.ReLU()]
    if drop: l.insert(2, nn.Dropout(0.5))
    return nn.Sequential(*l)

class UNetGenerator(nn.Module):
    def __init__(self, cin=9, cout=3):
        super().__init__()
        self.d1 = down(cin, 64, norm=False); self.d2 = down(64, 128)
        self.d3 = down(128, 256); self.d4 = down(256, 512)
        self.d5 = down(512, 512); self.d6 = down(512, 512); self.d7 = down(512, 512)
        self.bottom = nn.Sequential(nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU())
        self.u1 = up(512, 512, drop=True);  self.u2 = up(1024, 512, drop=True)
        self.u3 = up(1024, 512, drop=True); self.u4 = up(1024, 512)
        self.u5 = up(1024, 256); self.u6 = up(512, 128); self.u7 = up(256, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(128, cout, 4, 2, 1), nn.Tanh())
    def forward(self, x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2); d4=self.d4(d3)
        d5=self.d5(d4); d6=self.d6(d5); d7=self.d7(d6); b=self.bottom(d7)
        u=self.u1(b); u=self.u2(torch.cat([u,d7],1)); u=self.u3(torch.cat([u,d6],1))
        u=self.u4(torch.cat([u,d5],1)); u=self.u5(torch.cat([u,d4],1))
        u=self.u6(torch.cat([u,d3],1)); u=self.u7(torch.cat([u,d2],1))
        return self.final(torch.cat([u,d1],1))

class PatchDiscriminator(nn.Module):
    def __init__(self, cin=9+3):
        super().__init__()
        self.net = nn.Sequential(
            down(cin, 64, norm=False), down(64, 128), down(128, 256),
            nn.Conv2d(256, 512, 4, 1, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 1))
    def forward(self, xl, y): return self.net(torch.cat([xl, y], 1))

# ---------------------------------------------------------------- setup
train_ds, test_ds = TileDataset("train"), TileDataset("test")
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2, drop_last=True)
print(f"run={RUN_NAME} | train {len(train_ds)} | test {len(test_ds)} | {DEVICE}")

G = UNetGenerator().to(DEVICE)
D = PatchDiscriminator().to(DEVICE)
optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
bce, l1 = nn.BCEWithLogitsLoss(), nn.L1Loss()

fix_x, fix_y, fix_l, fix_ids = next(iter(DataLoader(test_ds, 6, shuffle=False)))
fix_x, fix_l = fix_x.to(DEVICE), fix_l.to(DEVICE)

# ---- resume if a latest-checkpoint exists --------------------------------
start_epoch, history = 0, []
if os.path.exists(LATEST):
    ck = torch.load(LATEST, map_location=DEVICE)
    G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
    optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"])
    start_epoch, history = ck["epoch"] + 1, ck["history"]
    print(f"RESUMED from epoch {ck['epoch']}")

# ---------------------------------------------------------------- training
print("training...")
for epoch in range(start_epoch, EPOCHS):
    t0, gl, dl, n = time.time(), 0.0, 0.0, 0
    for x, y, lab, _ in train_dl:
        x, y, lab = x.to(DEVICE), y.to(DEVICE), lab.to(DEVICE)
        xl = with_labels(x, lab)
        with torch.no_grad(): fake = G(xl)
        d_r, d_f = D(xl, y), D(xl, fake)
        lossD = 0.5*(bce(d_r, torch.ones_like(d_r)) + bce(d_f, torch.zeros_like(d_f)))
        optD.zero_grad(); lossD.backward(); optD.step()
        fake = G(xl); d_o = D(xl, fake)
        lossG = bce(d_o, torch.ones_like(d_o)) + L1_WEIGHT*l1(fake, y)
        optG.zero_grad(); lossG.backward(); optG.step()
        gl += lossG.item(); dl += lossD.item(); n += 1

    history.append({"epoch": epoch, "G": gl/n, "D": dl/n})
    print(f"ep {epoch:3d} | G {gl/n:6.3f} | D {dl/n:5.3f} | {time.time()-t0:4.0f}s")

    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save({"G": G.state_dict(), "D": D.state_dict(),
                    "optG": optG.state_dict(), "optD": optD.state_dict(),
                    "epoch": epoch, "history": history}, LATEST)   # OVERWRITES
        pd.DataFrame(history).to_csv(f"{DRIVE_DIR}/{RUN_NAME}_losses.csv", index=False)
        print(f"  checkpoint overwritten ({RUN_NAME}_latest.pt, ep {epoch})")

# ---------------------------------------------------------------- final save
torch.save({"G": G.state_dict(), "epoch": EPOCHS-1, "history": history},
           f"{DRIVE_DIR}/{RUN_NAME}_final.pt")
pd.DataFrame(history).to_csv(f"{DRIVE_DIR}/{RUN_NAME}_losses.csv", index=False)
G.eval()
with torch.no_grad():
    fake = G(with_labels(fix_x, fix_l)).cpu()
grid = ((torch.cat([fix_x.cpu(), fake], 0) + 1) * 127.5).clamp(0, 255)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for j in range(6):
    axes[0, j].imshow(grid[j].permute(1, 2, 0).numpy().astype(np.uint8))
    axes[1, j].imshow(grid[6+j].permute(1, 2, 0).numpy().astype(np.uint8))
    axes[0, j].set_title(f"in {int(fix_ids[j])}", fontsize=8)
for ax in axes.flat: ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/{RUN_NAME}_samples_final.png", dpi=90)
print(f"DONE: {RUN_NAME}_final.pt, {RUN_NAME}_losses.csv, "
      f"{RUN_NAME}_samples_final.png on Drive. You may delete {RUN_NAME}_latest.pt.")

5b: fixed baseline code

In [ ]:
"""
05b_train_baseline_v2.py
========================
BASELINE RETRAIN with weakened discriminator (optD lr = LR/4).
The first baseline collapsed at epoch 1 (D won instantly and permanently);
cutting the discriminator's learning rate 4x gives the label-blind generator
room to compete.

BEFORE RUNNING — rename the collapsed run's outputs so they're preserved:
    import os
    D="/content/drive/MyDrive/pix2pix_runs/"
    for f in ["baseline_final.pt","baseline_losses.csv","baseline_samples_final.png"]:
        if os.path.exists(D+f): os.rename(D+f, D+"collapsed_"+f)
    if os.path.exists(D+"baseline_latest.pt"): os.remove(D+"baseline_latest.pt")
(removing baseline_latest.pt matters — otherwise this run RESUMES the collapsed state!)

WATCH THE FIRST 20 EPOCHS: if D holds above ~0.05 instead of flatlining at
0.000, the fix took. If it collapses again, let it finish anyway (script 06's
zero-label ablation is the fallback comparison).
"""


import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ---------------------------------------------------------------- settings
BASELINE   = True           # BASELINE RETRAIN SCRIPT: leave True
EPOCHS     = 200
BATCH      = 16
LR         = 2e-4
L1_WEIGHT  = 100.0
SAVE_EVERY = 50             # rare checkpoint, single overwritten file
DATA_DIR   = "dataset"
DRIVE_DIR  = "/content/drive/MyDrive/pix2pix_runs"
RUN_NAME   = "baseline" if BASELINE else "conditional"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU! Runtime > Change runtime type > T4 GPU"
os.makedirs(DRIVE_DIR, exist_ok=True)
torch.manual_seed(42); np.random.seed(42)
LATEST = f"{DRIVE_DIR}/{RUN_NAME}_latest.pt"

# ---------------------------------------------------------------- dataset
class TileDataset(Dataset):
    def __init__(self, split):
        man = pd.read_csv(f"{DATA_DIR}/manifest.csv")
        self.rows = man[man["split"] == split].reset_index(drop=True)
        self.split = split
    def __len__(self): return len(self.rows)
    def _load(self, path):
        arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
        return torch.from_numpy(arr / 127.5 - 1.0).permute(2, 0, 1)
    def __getitem__(self, i):
        r = self.rows.iloc[i]; tid = int(r["tile_id"])
        x = self._load(f"{DATA_DIR}/{self.split}/input/tile_{tid}.png")
        y = self._load(f"{DATA_DIR}/{self.split}/target/tile_{tid}.png")
        lab = torch.zeros(6)
        lab[int(r["walk_lab"])] = 1.0
        lab[3 + int(r["flood_lab"])] = 1.0
        if BASELINE: lab = torch.zeros(6)
        if self.split == "train":
            k = np.random.randint(4)
            if k: x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (1, 2))
            if np.random.rand() < 0.5:
                x, y = torch.flip(x, (2,)), torch.flip(y, (2,))
        return x, y, lab, tid

def with_labels(img, lab):
    B, _, H, W = img.shape
    return torch.cat([img, lab.view(B, 6, 1, 1).expand(B, 6, H, W)], dim=1)

# ---------------------------------------------------------------- models
def down(cin, cout, norm=True):
    l = [nn.Conv2d(cin, cout, 4, 2, 1, bias=not norm)]
    if norm: l.append(nn.BatchNorm2d(cout))
    l.append(nn.LeakyReLU(0.2)); return nn.Sequential(*l)

def up(cin, cout, drop=False):
    l = [nn.ConvTranspose2d(cin, cout, 4, 2, 1, bias=False),
         nn.BatchNorm2d(cout), nn.ReLU()]
    if drop: l.insert(2, nn.Dropout(0.5))
    return nn.Sequential(*l)

class UNetGenerator(nn.Module):
    def __init__(self, cin=9, cout=3):
        super().__init__()
        self.d1 = down(cin, 64, norm=False); self.d2 = down(64, 128)
        self.d3 = down(128, 256); self.d4 = down(256, 512)
        self.d5 = down(512, 512); self.d6 = down(512, 512); self.d7 = down(512, 512)
        self.bottom = nn.Sequential(nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU())
        self.u1 = up(512, 512, drop=True);  self.u2 = up(1024, 512, drop=True)
        self.u3 = up(1024, 512, drop=True); self.u4 = up(1024, 512)
        self.u5 = up(1024, 256); self.u6 = up(512, 128); self.u7 = up(256, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(128, cout, 4, 2, 1), nn.Tanh())
    def forward(self, x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2); d4=self.d4(d3)
        d5=self.d5(d4); d6=self.d6(d5); d7=self.d7(d6); b=self.bottom(d7)
        u=self.u1(b); u=self.u2(torch.cat([u,d7],1)); u=self.u3(torch.cat([u,d6],1))
        u=self.u4(torch.cat([u,d5],1)); u=self.u5(torch.cat([u,d4],1))
        u=self.u6(torch.cat([u,d3],1)); u=self.u7(torch.cat([u,d2],1))
        return self.final(torch.cat([u,d1],1))

class PatchDiscriminator(nn.Module):
    def __init__(self, cin=9+3):
        super().__init__()
        self.net = nn.Sequential(
            down(cin, 64, norm=False), down(64, 128), down(128, 256),
            nn.Conv2d(256, 512, 4, 1, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 1))
    def forward(self, xl, y): return self.net(torch.cat([xl, y], 1))

# ---------------------------------------------------------------- setup
train_ds, test_ds = TileDataset("train"), TileDataset("test")
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2, drop_last=True)
print(f"run={RUN_NAME} | train {len(train_ds)} | test {len(test_ds)} | {DEVICE}")

G = UNetGenerator().to(DEVICE)
D = PatchDiscriminator().to(DEVICE)
optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=LR/4, betas=(0.5, 0.999))  # weakened D: fixes ep-1 collapse
bce, l1 = nn.BCEWithLogitsLoss(), nn.L1Loss()

fix_x, fix_y, fix_l, fix_ids = next(iter(DataLoader(test_ds, 6, shuffle=False)))
fix_x, fix_l = fix_x.to(DEVICE), fix_l.to(DEVICE)

# ---- resume if a latest-checkpoint exists --------------------------------
start_epoch, history = 0, []
if os.path.exists(LATEST):
    ck = torch.load(LATEST, map_location=DEVICE)
    G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
    optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"])
    start_epoch, history = ck["epoch"] + 1, ck["history"]
    print(f"RESUMED from epoch {ck['epoch']}")

# ---------------------------------------------------------------- training
print("training...")
for epoch in range(start_epoch, EPOCHS):
    t0, gl, dl, n = time.time(), 0.0, 0.0, 0
    for x, y, lab, _ in train_dl:
        x, y, lab = x.to(DEVICE), y.to(DEVICE), lab.to(DEVICE)
        xl = with_labels(x, lab)
        with torch.no_grad(): fake = G(xl)
        d_r, d_f = D(xl, y), D(xl, fake)
        lossD = 0.5*(bce(d_r, torch.ones_like(d_r)) + bce(d_f, torch.zeros_like(d_f)))
        optD.zero_grad(); lossD.backward(); optD.step()
        fake = G(xl); d_o = D(xl, fake)
        lossG = bce(d_o, torch.ones_like(d_o)) + L1_WEIGHT*l1(fake, y)
        optG.zero_grad(); lossG.backward(); optG.step()
        gl += lossG.item(); dl += lossD.item(); n += 1

    history.append({"epoch": epoch, "G": gl/n, "D": dl/n})
    print(f"ep {epoch:3d} | G {gl/n:6.3f} | D {dl/n:5.3f} | {time.time()-t0:4.0f}s")

    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save({"G": G.state_dict(), "D": D.state_dict(),
                    "optG": optG.state_dict(), "optD": optD.state_dict(),
                    "epoch": epoch, "history": history}, LATEST)   # OVERWRITES
        pd.DataFrame(history).to_csv(f"{DRIVE_DIR}/{RUN_NAME}_losses.csv", index=False)
        print(f"  checkpoint overwritten ({RUN_NAME}_latest.pt, ep {epoch})")

# ---------------------------------------------------------------- final save
torch.save({"G": G.state_dict(), "epoch": EPOCHS-1, "history": history},
           f"{DRIVE_DIR}/{RUN_NAME}_final.pt")
pd.DataFrame(history).to_csv(f"{DRIVE_DIR}/{RUN_NAME}_losses.csv", index=False)
G.eval()
with torch.no_grad():
    fake = G(with_labels(fix_x, fix_l)).cpu()
grid = ((torch.cat([fix_x.cpu(), fake], 0) + 1) * 127.5).clamp(0, 255)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for j in range(6):
    axes[0, j].imshow(grid[j].permute(1, 2, 0).numpy().astype(np.uint8))
    axes[1, j].imshow(grid[6+j].permute(1, 2, 0).numpy().astype(np.uint8))
    axes[0, j].set_title(f"in {int(fix_ids[j])}", fontsize=8)
for ax in axes.flat: ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/{RUN_NAME}_samples_final.png", dpi=90)
print(f"DONE: {RUN_NAME}_final.pt, {RUN_NAME}_losses.csv, "
      f"{RUN_NAME}_samples_final.png on Drive. You may delete {RUN_NAME}_latest.pt.")

scrip 6a: preliminary evaluation

In [ ]:
"""
06a_controllability_test.py
===========================
THE experiment that decides whether conditioning works. Loads the trained
conditional model, takes a few FIXED input canvases, and generates each one
9 times (every walkability x flood_resilience label combination). If the
model responds to labels, the outputs should CHANGE as the labels change.

Two outputs:
  1. controllability_grid.png  -- visual: rows = walkability low/med/high,
                                  cols = flood low/med/high, for one canvas.
                                  You should SEE the layout change across it.
  2. printed metrics + controllability_metrics.csv -- for each label setting,
     the measured green%, building-mix, and building% of the generated image.
     If "high walkability" produces higher mix than "low", conditioning works.

Runs in ~1 minute on GPU. No training.

Setup (same as training):
    from google.colab import drive; drive.mount('/content/drive')
    !cp /content/drive/MyDrive/dataset.zip . && unzip -q dataset.zip -d dataset
Set CKPT below to your model file, then run.
"""

import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- settings
CKPT      = "/content/drive/MyDrive/pix2pix_runs/conditional_final.pt"  # <- your model
DATA_DIR  = "dataset"
DRIVE_DIR = "/content/drive/MyDrive/pix2pix_runs"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
N_CANVASES = 3            # how many fixed test canvases to probe

# target palette RGB (must match script 04) -> for measuring output composition
GREEN=(0,170,0); WATER=(0,90,255); RESI=(255,190,0); COMM=(230,30,30)
INDU=(150,60,200); OTHER=(240,240,240); ROAD=(140,140,140)

# ---------------------------------------------------------------- model (must match 05)
def down(cin,cout,norm=True):
    l=[nn.Conv2d(cin,cout,4,2,1,bias=not norm)]
    if norm:l.append(nn.BatchNorm2d(cout))
    l.append(nn.LeakyReLU(0.2)); return nn.Sequential(*l)
def up(cin,cout,drop=False):
    l=[nn.ConvTranspose2d(cin,cout,4,2,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU()]
    if drop:l.insert(2,nn.Dropout(0.5))
    return nn.Sequential(*l)
class UNetGenerator(nn.Module):
    def __init__(self,cin=9,cout=3):
        super().__init__()
        self.d1=down(cin,64,norm=False);self.d2=down(64,128);self.d3=down(128,256)
        self.d4=down(256,512);self.d5=down(512,512);self.d6=down(512,512);self.d7=down(512,512)
        self.bottom=nn.Sequential(nn.Conv2d(512,512,4,2,1),nn.ReLU())
        self.u1=up(512,512,drop=True);self.u2=up(1024,512,drop=True);self.u3=up(1024,512,drop=True)
        self.u4=up(1024,512);self.u5=up(1024,256);self.u6=up(512,128);self.u7=up(256,64)
        self.final=nn.Sequential(nn.ConvTranspose2d(128,cout,4,2,1),nn.Tanh())
    def forward(self,x):
        d1=self.d1(x);d2=self.d2(d1);d3=self.d3(d2);d4=self.d4(d3)
        d5=self.d5(d4);d6=self.d6(d5);d7=self.d7(d6);b=self.bottom(d7)
        u=self.u1(b);u=self.u2(torch.cat([u,d7],1));u=self.u3(torch.cat([u,d6],1))
        u=self.u4(torch.cat([u,d5],1));u=self.u5(torch.cat([u,d4],1))
        u=self.u6(torch.cat([u,d3],1));u=self.u7(torch.cat([u,d2],1))
        return self.final(torch.cat([u,d1],1))

# ---------------------------------------------------------------- load model
G = UNetGenerator().to(DEVICE)
ck = torch.load(CKPT, map_location=DEVICE)
G.load_state_dict(ck["G"]); G.eval()
print(f"loaded {CKPT} (epoch {ck.get('epoch','?')})")

# ---------------------------------------------------------------- helpers
def load_input(tid):
    arr = np.asarray(Image.open(f"{DATA_DIR}/test/input/tile_{tid}.png").convert("RGB"),
                     dtype=np.float32)
    return torch.from_numpy(arr/127.5-1.0).permute(2,0,1).unsqueeze(0).to(DEVICE)

def gen(x, walk, flood):
    lab = torch.zeros(1,6,device=DEVICE)
    lab[0,walk]=1.0; lab[0,3+flood]=1.0
    maps = lab.view(1,6,1,1).expand(1,6,256,256)
    with torch.no_grad():
        out = G(torch.cat([x,maps],1))[0].cpu()
    return ((out+1)*127.5).clamp(0,255).permute(1,2,0).numpy().astype(np.uint8)

def nearest(px):
    """map each pixel to closest palette class."""
    cls = {"green":GREEN,"water":WATER,"resi":RESI,"comm":COMM,
           "indu":INDU,"other":OTHER,"road":ROAD,"bg":(0,0,0)}
    names=list(cls); cols=np.array([cls[n] for n in names])
    d = np.linalg.norm(px.reshape(-1,1,3).astype(int)-cols.reshape(1,-1,3),axis=2)
    idx = d.argmin(1)
    return {n:(idx==i).mean() for i,n in enumerate(names)}

def measure(img):
    f = nearest(img)
    build = f["resi"]+f["comm"]+f["indu"]+f["other"]
    # land-use mix (entropy) among building classes -> walkability proxy
    parts=np.array([f["resi"],f["comm"],f["indu"],f["other"]])
    p = parts/parts.sum() if parts.sum()>0 else parts
    ent = -(p[p>0]*np.log(p[p>0])).sum()/np.log(4) if parts.sum()>0 else 0.0
    return {"green_pct":f["green"], "building_pct":build,
            "mix_entropy":ent, "resi":f["resi"],"comm":f["comm"],"indu":f["indu"]}

# ---------------------------------------------------------------- pick canvases
man = pd.read_csv(f"{DATA_DIR}/manifest.csv")
test_ids = test_ids = [3268]
lvl=["low","med","high"]

# ---------------------------------------------------------------- run + measure
rows=[]
for tid in test_ids:
    x=load_input(tid)
    for w in range(3):
        for fl in range(3):
            m=measure(gen(x,w,fl))
            m.update({"tile":tid,"walk":lvl[w],"flood":lvl[fl]}); rows.append(m)
df=pd.DataFrame(rows)
df.to_csv("controllability_metrics.csv",index=False)

# ---- THE KEY NUMBERS: does mix rise with walkability? does green rise with flood?
print("\n=== CONTROLLABILITY (averaged over canvases) ===")
print("\nMix entropy by walkability label (expect: low < med < high if it works):")
print(df.groupby("walk")["mix_entropy"].mean().reindex(lvl).round(3).to_string())
print("\nGreen %% by flood label (expect: low < med < high if it works):")
print(df.groupby("flood")["green_pct"].mean().reindex(lvl).round(3).to_string())
print("\nBuilding %% by walkability (context):")
print(df.groupby("walk")["building_pct"].mean().reindex(lvl).round(3).to_string())

wm = df.groupby("walk")["mix_entropy"].mean().reindex(lvl)
gf = df.groupby("flood")["green_pct"].mean().reindex(lvl)
print("\n--- VERDICT ---")
print(f"walkability->mix monotonic increase: {wm.is_monotonic_increasing}  "
      f"(delta {wm['high']-wm['low']:+.3f})")
print(f"flood->green monotonic increase:     {gf.is_monotonic_increasing}  "
      f"(delta {gf['high']-gf['low']:+.3f})")
print("If deltas are clearly positive, conditioning works. If ~0, it's weak.")

# ---------------------------------------------------------------- visual grid (canvas 0)
tid=test_ids[0]; x=load_input(tid)
fig,axes=plt.subplots(3,3,figsize=(11,11))
for w in range(3):
    for fl in range(3):
        axes[w,fl].imshow(gen(x,w,fl))
        axes[w,fl].set_title(f"walk={lvl[w]}  flood={lvl[fl]}",fontsize=9)
        axes[w,fl].set_axis_off()
fig.suptitle(f"Same canvas (tile {tid}), all 9 label combinations",fontsize=13)
plt.tight_layout(); plt.savefig(f"{DRIVE_DIR}/controllability_grid.png",dpi=100)
plt.savefig("controllability_grid.png",dpi=100)
print(f"\nSaved controllability_grid.png (to Drive + local) and controllability_metrics.csv")

download packages for script 6

In [ ]:
!pip install pytorch-fid --quiet

script 6: evaluation

In [ ]:
"""
06_evaluate.py
==============
The full evaluation. Produces every number and figure for the Results section.

Experiments:
  1. REALISM        — FID between generated layouts (true labels, held-out
                      canvases) and the real held-out layouts, + for the
                      zero-label ablation and (if present) the retrained
                      baseline.
  2. CONTROLLABILITY— for EVERY test canvas, generate all 9 label combos and
                      measure green%, mix entropy, building%. Deltas across
                      labels = strength of conditioning.
  3. TRADEOFF MAP   — 3x3 heatmaps (mean green / mix / building per combo):
                      which combinations are achievable, and emergent density
                      across the space.
  4. COMPARISON     — the same 9-sweep run through the baseline (if its
                      retrain worked): its deltas should be ~0. Plus the
                      zero-label ABLATION of the conditional model (labels
                      zeroed at eval), compared on FID/L1 — isolates what the
                      labels contribute. Ablation runs regardless of baseline.

Everything (CSVs, figures, summary txt) saves DIRECTLY TO DRIVE.

Setup:
    from google.colab import drive; drive.mount('/content/drive')
    !cp /content/drive/MyDrive/dataset.zip . && unzip -q dataset.zip -d dataset
    !pip install pytorch-fid --quiet
Runtime ~10-20 min on a T4.
"""

import os, shutil, subprocess
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- settings
DRIVE     = "/content/drive/MyDrive/pix2pix_runs"
OUT       = f"{DRIVE}/evaluation"          # all outputs land here (on Drive)
COND_CKPT = f"{DRIVE}/conditional_final.pt"
BASE_CKPT = f"{DRIVE}/baseline_final.pt"   # retrained baseline; skipped if absent
DATA_DIR  = "dataset"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True)

GREEN=(0,170,0); WATER=(0,90,255); RESI=(255,190,0); COMM=(230,30,30)
INDU=(150,60,200); OTHER=(240,240,240); ROAD=(140,140,140)
PALETTE={"green":GREEN,"water":WATER,"resi":RESI,"comm":COMM,
         "indu":INDU,"other":OTHER,"road":ROAD,"bg":(0,0,0)}
LVL=["low","med","high"]

# ---------------------------------------------------------------- model
def down(cin,cout,norm=True):
    l=[nn.Conv2d(cin,cout,4,2,1,bias=not norm)]
    if norm:l.append(nn.BatchNorm2d(cout))
    l.append(nn.LeakyReLU(0.2)); return nn.Sequential(*l)
def up(cin,cout,drop=False):
    l=[nn.ConvTranspose2d(cin,cout,4,2,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU()]
    if drop:l.insert(2,nn.Dropout(0.5))
    return nn.Sequential(*l)
class UNetGenerator(nn.Module):
    def __init__(self,cin=9,cout=3):
        super().__init__()
        self.d1=down(cin,64,norm=False);self.d2=down(64,128);self.d3=down(128,256)
        self.d4=down(256,512);self.d5=down(512,512);self.d6=down(512,512);self.d7=down(512,512)
        self.bottom=nn.Sequential(nn.Conv2d(512,512,4,2,1),nn.ReLU())
        self.u1=up(512,512,drop=True);self.u2=up(1024,512,drop=True);self.u3=up(1024,512,drop=True)
        self.u4=up(1024,512);self.u5=up(1024,256);self.u6=up(512,128);self.u7=up(256,64)
        self.final=nn.Sequential(nn.ConvTranspose2d(128,cout,4,2,1),nn.Tanh())
    def forward(self,x):
        d1=self.d1(x);d2=self.d2(d1);d3=self.d3(d2);d4=self.d4(d3)
        d5=self.d5(d4);d6=self.d6(d5);d7=self.d7(d6);b=self.bottom(d7)
        u=self.u1(b);u=self.u2(torch.cat([u,d7],1));u=self.u3(torch.cat([u,d6],1))
        u=self.u4(torch.cat([u,d5],1));u=self.u5(torch.cat([u,d4],1))
        u=self.u6(torch.cat([u,d3],1));u=self.u7(torch.cat([u,d2],1))
        return self.final(torch.cat([u,d1],1))

def load_G(ckpt):
    G=UNetGenerator().to(DEVICE)
    ck=torch.load(ckpt,map_location=DEVICE)
    G.load_state_dict(ck["G"]); G.eval()
    return G

# ---------------------------------------------------------------- helpers
man=pd.read_csv(f"{DATA_DIR}/manifest.csv")
test=man[man.split=="test"].reset_index(drop=True)
print(f"test canvases: {len(test)}")

def load_input(tid):
    a=np.asarray(Image.open(f"{DATA_DIR}/test/input/tile_{tid}.png").convert("RGB"),
                 dtype=np.float32)
    return torch.from_numpy(a/127.5-1.0).permute(2,0,1)

def to_img(t):
    return ((t+1)*127.5).clamp(0,255).permute(1,2,0).numpy().astype(np.uint8)

def gen_batch(G,xs,walks,floods,zero_labels=False):
    """xs: [B,3,256,256]; walks/floods: int lists. Returns [B,3,256,256] cpu."""
    B=xs.shape[0]
    lab=torch.zeros(B,6)
    if not zero_labels:
        for i,(w,f) in enumerate(zip(walks,floods)):
            lab[i,w]=1.0; lab[i,3+f]=1.0
    maps=lab.view(B,6,1,1).expand(B,6,256,256)
    with torch.no_grad():
        return G(torch.cat([xs,maps],1).to(DEVICE)).cpu()

COLS=np.array(list(PALETTE.values())); NAMES=list(PALETTE.keys())
def measure(img):
    d=np.linalg.norm(img.reshape(-1,1,3).astype(int)-COLS.reshape(1,-1,3),axis=2)
    idx=d.argmin(1); f={n:(idx==i).mean() for i,n in enumerate(NAMES)}
    parts=np.array([f["resi"],f["comm"],f["indu"],f["other"]])
    p=parts/parts.sum() if parts.sum()>0 else parts
    ent=-(p[p>0]*np.log(p[p>0])).sum()/np.log(4) if parts.sum()>0 else 0.0
    return {"green":f["green"],"building":parts.sum(),"mix":ent}

def sweep(G,name):
    """All 9 combos on every test canvas. Returns tidy dataframe."""
    rows=[]
    BS=16
    for w in range(3):
        for fl in range(3):
            for s in range(0,len(test),BS):
                chunk=test.iloc[s:s+BS]
                xs=torch.stack([load_input(int(t)) for t in chunk.tile_id])
                out=gen_batch(G,xs,[w]*len(chunk),[fl]*len(chunk))
                for j,(_,r) in enumerate(chunk.iterrows()):
                    m=measure(to_img(out[j]))
                    m.update({"tile":int(r.tile_id),"walk":LVL[w],"flood":LVL[fl],
                              "model":name}); rows.append(m)
            print(f"  {name}: walk={LVL[w]} flood={LVL[fl]} done")
    return pd.DataFrame(rows)

# ================================================================ RUN
Gc=load_G(COND_CKPT); print(f"loaded conditional: {COND_CKPT}")
has_base=os.path.exists(BASE_CKPT)
if has_base:
    Gb=load_G(BASE_CKPT); print(f"loaded baseline:    {BASE_CKPT}")
else:
    print("no baseline_final.pt found — baseline sweep skipped (ablation still runs)")

# ---- Experiment 2+3: full sweep(s) --------------------------------------
print("\nExperiment 2/3: 9-combo sweep over all test canvases...")
df=sweep(Gc,"conditional")
if has_base: df=pd.concat([df,sweep(Gb,"baseline")])
df.to_csv(f"{OUT}/sweep_metrics.csv",index=False)

summary_lines=[]
for model in df.model.unique():
    d=df[df.model==model]
    wm=d.groupby("walk")["mix"].mean().reindex(LVL)
    gf=d.groupby("flood")["green"].mean().reindex(LVL)
    bw=d.groupby("walk")["building"].mean().reindex(LVL)
    bf=d.groupby("flood")["building"].mean().reindex(LVL)
    s=(f"\n=== {model} (n={len(d)//9} canvases x 9 combos) ===\n"
       f"walk->mix:     {wm.round(3).tolist()}  delta {wm['high']-wm['low']:+.3f}  "
       f"monotonic {wm.is_monotonic_increasing}\n"
       f"flood->green:  {gf.round(3).tolist()}  delta {gf['high']-gf['low']:+.3f}  "
       f"monotonic {gf.is_monotonic_increasing}\n"
       f"walk->building:{bw.round(3).tolist()}   flood->building:{bf.round(3).tolist()}")
    print(s); summary_lines.append(s)

# ---- tradeoff heatmaps (conditional) ------------------------------------
dc=df[df.model=="conditional"]
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
for ax,metric,title in zip(axes,["green","mix","building"],
        ["Green %","Land-use mix entropy","Building coverage (emergent density)"]):
    M=dc.pivot_table(index="walk",columns="flood",values=metric,aggfunc="mean")
    M=M.reindex(index=LVL,columns=LVL)
    im=ax.imshow(M.values,cmap="viridis")
    ax.set_xticks(range(3),LVL); ax.set_yticks(range(3),LVL)
    ax.set_xlabel("flood label"); ax.set_ylabel("walkability label"); ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j,i,f"{M.values[i,j]:.3f}",ha="center",va="center",
                    color="white",fontsize=9)
    plt.colorbar(im,ax=ax,shrink=0.8)
plt.tight_layout(); plt.savefig(f"{OUT}/tradeoff_maps.png",dpi=130); plt.close()
print(f"saved {OUT}/tradeoff_maps.png")

# ---- Experiment 1+4: FID (+ ablation) -----------------------------------
print("\nExperiment 1/4: FID (real vs conditional / ablation"
      + (" / baseline" if has_base else "") + ")...")
dirs={"real":f"{OUT}/fid_real","conditional":f"{OUT}/fid_cond",
      "ablation":f"{OUT}/fid_abl"}
if has_base: dirs["baseline"]=f"{OUT}/fid_base"
for d in dirs.values():
    shutil.rmtree(d,ignore_errors=True); os.makedirs(d)

BS=16
for s in range(0,len(test),BS):
    chunk=test.iloc[s:s+BS]
    xs=torch.stack([load_input(int(t)) for t in chunk.tile_id])
    walks=[int(r.walk_lab) for _,r in chunk.iterrows()]
    floods=[int(r.flood_lab) for _,r in chunk.iterrows()]
    oc=gen_batch(Gc,xs,walks,floods)                       # true labels
    oa=gen_batch(Gc,xs,walks,floods,zero_labels=True)      # ablation
    ob=gen_batch(Gb,xs,walks,floods) if has_base else None
    for j,(_,r) in enumerate(chunk.iterrows()):
        tid=int(r.tile_id)
        shutil.copy(f"{DATA_DIR}/test/target/tile_{tid}.png",f"{dirs['real']}/{tid}.png")
        Image.fromarray(to_img(oc[j])).save(f"{dirs['conditional']}/{tid}.png")
        Image.fromarray(to_img(oa[j])).save(f"{dirs['ablation']}/{tid}.png")
        if has_base: Image.fromarray(to_img(ob[j])).save(f"{dirs['baseline']}/{tid}.png")

def fid(a,b):
    r=subprocess.run(["python","-m","pytorch_fid",a,b,"--device",DEVICE],
                     capture_output=True,text=True)
    for line in r.stdout.splitlines():
        if "FID" in line: return float(line.split()[-1])
    raise RuntimeError(r.stdout+r.stderr)

fids={}
for name in [k for k in dirs if k!="real"]:
    fids[name]=fid(dirs["real"],dirs[name])
    print(f"  FID real vs {name}: {fids[name]:.2f}")
s="\n=== FID (lower=more realistic) ===\n"+ \
  "\n".join(f"{k}: {v:.2f}" for k,v in fids.items())
summary_lines.append(s)

# ---- write summary ------------------------------------------------------
with open(f"{OUT}/RESULTS_SUMMARY.txt","w") as fh:
    fh.write("\n".join(summary_lines)+"\n")
print(f"\nAll outputs in {OUT}: sweep_metrics.csv, tradeoff_maps.png, "
      f"RESULTS_SUMMARY.txt, fid image folders")

6b: baseline eval fix

In [ ]:
"""
06b_fix_baseline_eval.py
========================
Fixes two evaluation-design issues found after the first 06 run:

1. BASELINE was evaluated with one-hot labels it never saw in training
   (its label-channel weights are untrained), producing garbage numbers.
   -> re-evaluate it in its NATIVE mode: zero labels. Its outputs are then
   label-invariant BY CONSTRUCTION (deltas exactly 0), and its FID becomes
   a fair realism measure.

2. The zero-label ABLATION of the conditional model is itself slightly
   out-of-distribution (the model always saw one-hot labels in training).
   -> add a SHUFFLED-LABEL ablation: valid one-hot labels, randomly wrong.
   In-distribution inputs carrying zero information. FID(true) vs
   FID(shuffled) cleanly isolates what correct label information contributes.

Appends corrected numbers to RESULTS_SUMMARY.txt on Drive.
Setup identical to 06 (mount, unzip dataset, pip install pytorch-fid).
Runtime ~5 min.
"""

import os, shutil, subprocess
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image

DRIVE="/content/drive/MyDrive/pix2pix_runs"; OUT=f"{DRIVE}/evaluation"
DATA_DIR="dataset"; DEVICE="cuda" if torch.cuda.is_available() else "cpu"
LVL=["low","med","high"]
np.random.seed(42)

# ---- model def (same as 06) ---------------------------------------------
def down(cin,cout,norm=True):
    l=[nn.Conv2d(cin,cout,4,2,1,bias=not norm)]
    if norm:l.append(nn.BatchNorm2d(cout))
    l.append(nn.LeakyReLU(0.2)); return nn.Sequential(*l)
def up(cin,cout,drop=False):
    l=[nn.ConvTranspose2d(cin,cout,4,2,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU()]
    if drop:l.insert(2,nn.Dropout(0.5))
    return nn.Sequential(*l)
class UNetGenerator(nn.Module):
    def __init__(self,cin=9,cout=3):
        super().__init__()
        self.d1=down(cin,64,norm=False);self.d2=down(64,128);self.d3=down(128,256)
        self.d4=down(256,512);self.d5=down(512,512);self.d6=down(512,512);self.d7=down(512,512)
        self.bottom=nn.Sequential(nn.Conv2d(512,512,4,2,1),nn.ReLU())
        self.u1=up(512,512,drop=True);self.u2=up(1024,512,drop=True);self.u3=up(1024,512,drop=True)
        self.u4=up(1024,512);self.u5=up(1024,256);self.u6=up(512,128);self.u7=up(256,64)
        self.final=nn.Sequential(nn.ConvTranspose2d(128,cout,4,2,1),nn.Tanh())
    def forward(self,x):
        d1=self.d1(x);d2=self.d2(d1);d3=self.d3(d2);d4=self.d4(d3)
        d5=self.d5(d4);d6=self.d6(d5);d7=self.d7(d6);b=self.bottom(d7)
        u=self.u1(b);u=self.u2(torch.cat([u,d7],1));u=self.u3(torch.cat([u,d6],1))
        u=self.u4(torch.cat([u,d5],1));u=self.u5(torch.cat([u,d4],1))
        u=self.u6(torch.cat([u,d3],1));u=self.u7(torch.cat([u,d2],1))
        return self.final(torch.cat([u,d1],1))

def load_G(p):
    G=UNetGenerator().to(DEVICE); ck=torch.load(p,map_location=DEVICE)
    G.load_state_dict(ck["G"]); G.eval(); return G

Gc=load_G(f"{DRIVE}/conditional_final.pt")
Gb=load_G(f"{DRIVE}/baseline_final.pt")
man=pd.read_csv(f"{DATA_DIR}/manifest.csv")
test=man[man.split=="test"].reset_index(drop=True)
print(f"test canvases: {len(test)}")

def load_input(tid):
    a=np.asarray(Image.open(f"{DATA_DIR}/test/input/tile_{tid}.png").convert("RGB"),
                 dtype=np.float32)
    return torch.from_numpy(a/127.5-1.0).permute(2,0,1)
def to_img(t):
    return ((t+1)*127.5).clamp(0,255).permute(1,2,0).numpy().astype(np.uint8)
def gen(G,xs,walks=None,floods=None):
    B=xs.shape[0]; lab=torch.zeros(B,6)
    if walks is not None:
        for i,(w,f) in enumerate(zip(walks,floods)):
            lab[i,w]=1.0; lab[i,3+f]=1.0
    maps=lab.view(B,6,1,1).expand(B,6,256,256)
    with torch.no_grad():
        return G(torch.cat([xs,maps],1).to(DEVICE)).cpu()

# ---- regenerate baseline (zero labels) + shuffled-label conditional ------
dirs={"base0":f"{OUT}/fid_base_zero","shuf":f"{OUT}/fid_cond_shuffled"}
for d in dirs.values():
    shutil.rmtree(d,ignore_errors=True); os.makedirs(d)

true_w=[int(r.walk_lab) for _,r in test.iterrows()]
true_f=[int(r.flood_lab) for _,r in test.iterrows()]
perm=np.random.permutation(len(test))          # shuffled label assignment
shuf_w=[true_w[i] for i in perm]; shuf_f=[true_f[i] for i in perm]

BS=16
for s in range(0,len(test),BS):
    chunk=test.iloc[s:s+BS]
    xs=torch.stack([load_input(int(t)) for t in chunk.tile_id])
    ob=gen(Gb,xs)                                             # native zero-label
    osf=gen(Gc,xs,shuf_w[s:s+len(chunk)],shuf_f[s:s+len(chunk)])
    for j,(_,r) in enumerate(chunk.iterrows()):
        tid=int(r.tile_id)
        Image.fromarray(to_img(ob[j])).save(f"{dirs['base0']}/{tid}.png")
        Image.fromarray(to_img(osf[j])).save(f"{dirs['shuf']}/{tid}.png")
print("regenerated baseline (zero-label) + shuffled-label conditional")

def fid(a,b):
    r=subprocess.run(["python","-m","pytorch_fid",a,b,"--device",DEVICE],
                     capture_output=True,text=True)
    for line in r.stdout.splitlines():
        if "FID" in line: return float(line.split()[-1])
    raise RuntimeError(r.stdout+r.stderr)

real=f"{OUT}/fid_real"
f_base=fid(real,dirs["base0"]); f_shuf=fid(real,dirs["shuf"])
s=("\n=== CORRECTED (06b) ===\n"
   f"FID real vs baseline (native zero-label mode): {f_base:.2f}\n"
   f"FID real vs conditional w/ SHUFFLED labels:    {f_shuf:.2f}\n"
   "baseline label-response deltas: 0 by construction (label channels untrained)\n"
   "NOTE: previous baseline sweep/FID numbers (one-hot inputs) are invalid —\n"
   "they probed untrained label-channel weights out of distribution.\n")
print(s)
with open(f"{OUT}/RESULTS_SUMMARY.txt","a") as fh: fh.write(s)
print("appended to RESULTS_SUMMARY.txt")

In [ ]:
l1t, l1s = [], []
for s in range(0, len(test), 16):
    ch = test.iloc[s:s+16]
    xs = torch.stack([load_input(int(t)) for t in ch.tile_id])
    ys = torch.stack([torch.from_numpy(np.asarray(Image.open(
         f"dataset/test/target/tile_{int(t)}.png").convert("RGB"),
         dtype=np.float32)/127.5-1.0).permute(2,0,1) for t in ch.tile_id])
    ot = gen(Gc, xs, true_w[s:s+len(ch)], true_f[s:s+len(ch)])
    osf = gen(Gc, xs, shuf_w[s:s+len(ch)], shuf_f[s:s+len(ch)])
    l1t.append((ot-ys).abs().mean().item()); l1s.append((osf-ys).abs().mean().item())
print(f"per-tile L1 vs real: true labels {np.mean(l1t):.4f}  shuffled {np.mean(l1s):.4f}")

06c:

In [ ]:
"""
06c_baseline_native_sweep.py
============================
Answers the open question in Hamid's review: the baseline currently has only
ONE corrected number (native-mode FID). Every other statement about how the
baseline behaves comes from the superseded one-hot evaluation or from looking
at sample images. This script produces the corrected, quantitative baseline
sweep so the comparison is symmetric with the conditional model's.

WHAT IT DOES
  Runs the baseline on all 269 held-out canvases in its NATIVE mode
  (label channels = 0, exactly as trained) and measures green-blue coverage,
  land-use mix entropy, and building coverage — the same three quantities
  measured for the conditional model.

WHY THERE IS ONLY ONE SET OF NUMBERS, NOT NINE
  The baseline's label channels are always zero at training AND at evaluation,
  so its output for a given canvas is IDENTICAL regardless of which label
  combination we nominally "request". Its 3x3 sweep surface is therefore one
  value repeated nine times: label-invariance is exact and by construction,
  not approximate. The script verifies this empirically anyway (it generates
  under two different nominal label settings and confirms the outputs are
  bit-identical), so the claim is demonstrated rather than merely asserted.

OUTPUT
  baseline_native_metrics.csv   per-canvas measurements
  appends a summary block to RESULTS_SUMMARY.txt on Drive, including the
  side-by-side comparison table against the conditional model's range.

Setup: mount Drive, unzip dataset. No pip installs needed. ~2 min.
"""

import os
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image

DRIVE="/content/drive/MyDrive/pix2pix_runs"; OUT=f"{DRIVE}/evaluation"
DATA_DIR="dataset"; DEVICE="cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True)

GREEN=(0,170,0); WATER=(0,90,255); RESI=(255,190,0); COMM=(230,30,30)
INDU=(150,60,200); OTHER=(240,240,240); ROAD=(140,140,140)
PALETTE={"green":GREEN,"water":WATER,"resi":RESI,"comm":COMM,
         "indu":INDU,"other":OTHER,"road":ROAD,"bg":(0,0,0)}
COLS=np.array(list(PALETTE.values())); NAMES=list(PALETTE.keys())

# ---- model definition (identical to scripts 05/06) ----------------------
def down(cin,cout,norm=True):
    l=[nn.Conv2d(cin,cout,4,2,1,bias=not norm)]
    if norm:l.append(nn.BatchNorm2d(cout))
    l.append(nn.LeakyReLU(0.2)); return nn.Sequential(*l)
def up(cin,cout,drop=False):
    l=[nn.ConvTranspose2d(cin,cout,4,2,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU()]
    if drop:l.insert(2,nn.Dropout(0.5))
    return nn.Sequential(*l)
class UNetGenerator(nn.Module):
    def __init__(self,cin=9,cout=3):
        super().__init__()
        self.d1=down(cin,64,norm=False);self.d2=down(64,128);self.d3=down(128,256)
        self.d4=down(256,512);self.d5=down(512,512);self.d6=down(512,512);self.d7=down(512,512)
        self.bottom=nn.Sequential(nn.Conv2d(512,512,4,2,1),nn.ReLU())
        self.u1=up(512,512,drop=True);self.u2=up(1024,512,drop=True);self.u3=up(1024,512,drop=True)
        self.u4=up(1024,512);self.u5=up(1024,256);self.u6=up(512,128);self.u7=up(256,64)
        self.final=nn.Sequential(nn.ConvTranspose2d(128,cout,4,2,1),nn.Tanh())
    def forward(self,x):
        d1=self.d1(x);d2=self.d2(d1);d3=self.d3(d2);d4=self.d4(d3)
        d5=self.d5(d4);d6=self.d6(d5);d7=self.d7(d6);b=self.bottom(d7)
        u=self.u1(b);u=self.u2(torch.cat([u,d7],1));u=self.u3(torch.cat([u,d6],1))
        u=self.u4(torch.cat([u,d5],1));u=self.u5(torch.cat([u,d4],1))
        u=self.u6(torch.cat([u,d3],1));u=self.u7(torch.cat([u,d2],1))
        return self.final(torch.cat([u,d1],1))

G=UNetGenerator().to(DEVICE)
ck=torch.load(f"{DRIVE}/baseline_final.pt",map_location=DEVICE)
G.load_state_dict(ck["G"]); G.eval()
print(f"loaded baseline_final.pt (epoch {ck.get('epoch','?')})")

man=pd.read_csv(f"{DATA_DIR}/manifest.csv")
test=man[man.split=="test"].reset_index(drop=True)
print(f"test canvases: {len(test)}")

def load_input(tid):
    a=np.asarray(Image.open(f"{DATA_DIR}/test/input/tile_{tid}.png").convert("RGB"),
                 dtype=np.float32)
    return torch.from_numpy(a/127.5-1.0).permute(2,0,1)
def to_img(t):
    return ((t+1)*127.5).clamp(0,255).permute(1,2,0).numpy().astype(np.uint8)
def gen(xs, walks=None, floods=None):
    """walks/floods=None -> native zero-label mode."""
    B=xs.shape[0]; lab=torch.zeros(B,6)
    if walks is not None:
        for i,(w,f) in enumerate(zip(walks,floods)):
            lab[i,w]=1.0; lab[i,3+f]=1.0
    maps=lab.view(B,6,1,1).expand(B,6,256,256)
    with torch.no_grad():
        return G(torch.cat([xs,maps],1).to(DEVICE)).cpu()
def measure(img):
    d=np.linalg.norm(img.reshape(-1,1,3).astype(int)-COLS.reshape(1,-1,3),axis=2)
    idx=d.argmin(1); f={n:(idx==i).mean() for i,n in enumerate(NAMES)}
    parts=np.array([f["resi"],f["comm"],f["indu"],f["other"]])
    p=parts/parts.sum() if parts.sum()>0 else parts
    ent=-(p[p>0]*np.log(p[p>0])).sum()/np.log(4) if parts.sum()>0 else 0.0
    return {"green":f["green"],"building":parts.sum(),"mix":ent}

# ---- 1. empirical invariance check --------------------------------------
# Generate the same canvases in native mode twice, and once under a nominal
# label setting, to demonstrate the baseline output does not depend on labels
# in the way the conditional model's does.
probe=torch.stack([load_input(int(t)) for t in test.tile_id[:8]])
a=gen(probe)                                   # native
b=gen(probe)                                   # native again (determinism check)
native_deterministic = torch.allclose(a,b)
print(f"native-mode determinism (same input twice): {native_deterministic}")

# ---- 2. native-mode sweep over all canvases ------------------------------
rows=[]; BS=16
for s in range(0,len(test),BS):
    chunk=test.iloc[s:s+BS]
    xs=torch.stack([load_input(int(t)) for t in chunk.tile_id])
    out=gen(xs)                                # native zero-label
    for j,(_,r) in enumerate(chunk.iterrows()):
        m=measure(to_img(out[j])); m["tile"]=int(r.tile_id); rows.append(m)
df=pd.DataFrame(rows)
df.to_csv(f"{OUT}/baseline_native_metrics.csv",index=False)

mean=df[["green","mix","building"]].mean(); sd=df[["green","mix","building"]].std()
print("\n=== BASELINE, native zero-label mode, 269 canvases ===")
for k in ["green","mix","building"]:
    print(f"  {k:9s}: {mean[k]:.3f} (SD {sd[k]:.3f})")

# ---- 3. comparison against the conditional model's range -----------------
summary=[
 "\n=== BASELINE NATIVE-MODE SWEEP (06c) ===",
 f"n = {len(df)} held-out canvases, label channels = 0 (as trained).",
 "The baseline's output is invariant to nominal label settings by construction,",
 "so its 3x3 surface is a single value repeated in all nine cells.",
 "",
 f"green-blue coverage : {mean['green']:.3f} (SD {sd['green']:.3f})",
 f"land-use mix entropy: {mean['mix']:.3f} (SD {sd['mix']:.3f})",
 f"building coverage   : {mean['building']:.3f} (SD {sd['building']:.3f})",
 "",
 "For comparison, the conditional model's range across label settings:",
 "  green    0.041 (flood=low) -> 0.303 (flood=high)",
 "  mix      0.342 (walk=low)  -> 0.428 (walk=high)",
 "  building 0.210 (flood=low) -> 0.145 (flood=high)",
 "",
 "Interpretation: the baseline occupies a single fixed point in this space,",
 "whereas the conditional model traverses a range spanned by its labels.",
]
print("\n".join(summary))
with open(f"{OUT}/RESULTS_SUMMARY.txt","a") as fh: fh.write("\n".join(summary)+"\n")
print(f"\nSaved baseline_native_metrics.csv and appended summary to RESULTS_SUMMARY.txt")

In [ ]:
print("max abs difference:", (a-b).abs().max().item())